# 02 — Data Cleaning
## Nexora Supply Chain Analytics Platform

**Business objective:** transform the raw DataCo dataset into a consistent, privacy-aware, analytics-ready CSV for MySQL staging, star-schema modelling, and Tableau.

### Cleaning principles
1. Preserve the business grain: **one row = one order item**.
2. Never delete legitimate losses or operational outliers merely because they are unusual.
3. Remove direct personal/sensitive attributes not needed for analysis.
4. Standardize names, text, dates, and numeric types.
5. Resolve duplicates deterministically.
6. Add auditable features for delivery, profit, discount, and time analysis.
7. Validate the final export and reconcile row counts.

**Primary output:** `dataco_supply_chain_cleaned.csv`

## 1. Environment setup

In [ ]:
import importlib.util, subprocess, sys
required = ['pandas', 'numpy', 'openpyxl']
for package in required:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

import os
import re
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
print('Environment ready.')

Environment ready.


## 2. Load raw data and verify full import

In [ ]:
FILE_PATH = '/content/DataCoSupplyChainDataset.csv'
EXPECTED_MIN_ROWS = 180_000
EXPECTED_COLUMNS = 53


def resolve_input_file(file_path: str) -> str:
    if os.path.exists(file_path):
        return file_path
    try:
        from google.colab import files
        print('Upload DataCoSupplyChainDataset.csv ...')
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError('No file was uploaded.')
        return next(iter(uploaded.keys()))
    except ImportError as exc:
        raise FileNotFoundError(f'Set FILE_PATH to the valid CSV path: {file_path}') from exc


def read_csv_robust(file_path: str) -> pd.DataFrame:
    for encoding in ('latin-1', 'utf-8', 'cp1252'):
        try:
            return pd.read_csv(file_path, encoding=encoding, low_memory=False)
        except UnicodeDecodeError:
            continue
    raise RuntimeError('CSV encoding could not be detected.')

FILE_PATH = resolve_input_file(FILE_PATH)
df = read_csv_robust(FILE_PATH)
raw_rows, raw_cols = df.shape

print(f'Raw shape: {df.shape}')
assert raw_rows >= EXPECTED_MIN_ROWS, f'Only {raw_rows:,} rows loaded. Check the source CSV.'
assert raw_cols == EXPECTED_COLUMNS, f'Expected {EXPECTED_COLUMNS} columns, found {raw_cols}.'
display(df.head())

Upload DataCoSupplyChainDataset.csv ...


Saving DataCoSupplyChainDataset.csv to DataCoSupplyChainDataset.csv
Raw shape: (180519, 53)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.2500,314.6400,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0000,2,Fitness,18.2515,-66.0371,Pacific Asia,Bekasi,Indonesia,20755,1/31/2018 22:56,77202,1360,13.1100,0.0400,180517,327.7500,0.2900,1,327.7500,314.6400,91.2500,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.7500,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.0900,311.3600,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0000,2,Fitness,18.2795,-66.0371,Pacific Asia,Bikaner,India,19492,1/13/2018 12:27,75939,1360,16.3900,0.0500,179254,327.7500,-0.8000,1,327.7500,311.3600,-249.0900,South Asia,Rajastán,PENDING,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.7500,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.7800,309.7200,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,"95,125.0000",2,Fitness,37.2922,-121.8813,Pacific Asia,Bikaner,India,19491,1/13/2018 12:06,75938,1360,18.0300,0.0600,179253,327.7500,-0.8000,1,327.7500,309.7200,-247.7800,South Asia,Rajastán,CLOSED,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.7500,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.8600,304.8100,Advance shipping,0,73,Sporting Goods,Los Angeles,EE. UU.,XXXXXXXXX,Tana,19490,Tate,XXXXXXXXX,Home Office,CA,3200 Amber Bend,"90,027.0000",2,Fitness,34.1259,-118.2910,Pacific Asia,Townsville,Australia,19490,1/13/2018 11:45,75937,1360,22.9400,0.0700,179252,327.7500,0.0800,1,327.7500,304.8100,22.8600,Oceania,Queensland,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.7500,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.2100,298.2500,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Orli,19489,Hendricks,XXXXXXXXX,Corporate,PR,8671 Iron Anchor Corners,725.0000,2,Fitness,18.2538,-66.0370,Pacific Asia,Townsville,Australia,19489,1/13/2018 11:24,75936,1360,29.5000,0.0900,179251,327.7500,0.4500,1,327.7500,298.2500,134.2100,Oceania,Queensland,PENDING_PAYMENT,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.7500,0,1/15/2018 11:24,Standard Class


## 3. Initialize audit log
Every material transformation is recorded for documentation and reproducibility.

In [ ]:
audit_log = []

def log_step(step, rows_before, rows_after, columns_before, columns_after, note=''):
    audit_log.append({
        'step': step,
        'rows_before': int(rows_before),
        'rows_after': int(rows_after),
        'rows_removed': int(rows_before - rows_after),
        'columns_before': int(columns_before),
        'columns_after': int(columns_after),
        'note': note
    })

log_step('load_raw_data', raw_rows, len(df), raw_cols, df.shape[1], 'Raw CSV loaded successfully.')

## 4. Standardize column names
All names are converted to lower-case `snake_case`, making them safer for Python, MySQL, and Tableau.

In [ ]:
def to_snake_case(name: str) -> str:
    name = str(name).strip()
    name = re.sub(r'\([^)]*\)', '', name)  # remove parenthetical labels such as (DateOrders)
    name = name.replace('%', ' percent ')
    name = re.sub(r'[^A-Za-z0-9]+', '_', name)
    name = re.sub(r'_+', '_', name).strip('_').lower()
    return name

before_cols = df.shape[1]
original_to_clean = {c: to_snake_case(c) for c in df.columns}
df = df.rename(columns=original_to_clean)

# Explicit improvements for analytical clarity
rename_overrides = {
    'type': 'payment_type',
    'benefit_per_order': 'order_profit',
    'sales_per_customer': 'sales_per_customer',
    'late_delivery_risk': 'late_delivery_risk',
    'order_profit_per_order': 'order_profit_per_order'
}
df = df.rename(columns={k:v for k,v in rename_overrides.items() if k in df.columns})

assert not df.columns.duplicated().any(), 'Duplicate column names were created.'
log_step('standardize_column_names', len(df), len(df), before_cols, df.shape[1], 'Converted columns to snake_case.')

column_mapping = pd.DataFrame(original_to_clean.items(), columns=['raw_column', 'clean_column'])
display(column_mapping)
print(df.columns.tolist())

,raw_column,clean_column
0,Type,type
1,Days for shipping (real),days_for_shipping
2,Days for shipment (scheduled),days_for_shipment
3,Benefit per order,benefit_per_order
4,Sales per customer,sales_per_customer
5,Delivery Status,delivery_status
6,Late_delivery_risk,late_delivery_risk
7,Category Id,category_id
8,Category Name,category_name
9,Customer City,customer_city


['payment_type', 'days_for_shipping', 'days_for_shipment', 'order_profit', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_email', 'customer_fname', 'customer_id', 'customer_lname', 'customer_password', 'customer_segment', 'customer_state', 'customer_street', 'customer_zipcode', 'department_id', 'department_name', 'latitude', 'longitude', 'market', 'order_city', 'order_country', 'order_customer_id', 'order_date', 'order_id', 'order_item_cardprod_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_id', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'order_zipcode', 'product_card_id', 'product_category_id', 'product_description', 'product_image', 'product_name', 'product_price', 'product_status', 'shipping_date', 'shipping_mode']


## 5. Remove direct personal and non-analytical fields
The project retains stable customer IDs for aggregation but removes names, email, password, street address, and image URL. This reduces privacy exposure without losing analytical value.

In [ ]:
pii_and_unused = [
    'customer_email', 'customer_fname', 'customer_lname', 'customer_password',
    'customer_street', 'product_description', 'product_image'
]
existing_drop = [c for c in pii_and_unused if c in df.columns]
rows_before, cols_before = df.shape
df = df.drop(columns=existing_drop)
log_step('drop_pii_and_unused_columns', rows_before, len(df), cols_before, df.shape[1], f'Dropped: {existing_drop}')
print('Dropped columns:', existing_drop)
print('Current shape:', df.shape)

Dropped columns: ['customer_email', 'customer_fname', 'customer_lname', 'customer_password', 'customer_street', 'product_description', 'product_image']
Current shape: (180519, 46)


## 6. Normalize whitespace and blank strings
Only object/string fields are altered. Blank text becomes a true missing value.

In [ ]:
text_cols = df.select_dtypes(include=['object', 'string']).columns
for col in text_cols:
    df[col] = (
        df[col]
        .astype('string')
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
        .replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})
    )

log_step('normalize_text', len(df), len(df), df.shape[1], df.shape[1], 'Trimmed whitespace and normalized blank strings.')
print(f'Normalized {len(text_cols)} text columns.')

Normalized 18 text columns.


## 7. Parse dates and standardize numeric types

In [ ]:
date_cols = [c for c in ['order_date', 'shipping_date'] if c in df.columns]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

integer_like = [
    'late_delivery_risk', 'category_id', 'customer_id', 'department_id',
    'order_customer_id', 'order_id', 'order_item_cardprod_id', 'order_item_id',
    'order_item_quantity', 'product_card_id', 'product_category_id', 'product_status',
    'days_for_shipping_real', 'days_for_shipment_scheduled'
]
for col in [c for c in integer_like if c in df.columns]:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

numeric_like = [
    'order_profit', 'sales_per_customer', 'latitude', 'longitude',
    'order_item_discount', 'order_item_discount_rate', 'order_item_product_price',
    'order_item_profit_ratio', 'sales', 'order_item_total',
    'order_profit_per_order', 'product_price', 'customer_zipcode'
]
for col in [c for c in numeric_like if c in df.columns]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

log_step('standardize_data_types', len(df), len(df), df.shape[1], df.shape[1], 'Parsed dates and converted numeric fields.')
display(df.dtypes.rename('dtype').to_frame())

,dtype
payment_type,string[python]
days_for_shipping,int64
days_for_shipment,int64
order_profit,float64
sales_per_customer,float64
delivery_status,string[python]
late_delivery_risk,Int64
category_id,Int64
category_name,string[python]
customer_city,string[python]


## 8. Handle duplicates
Exact duplicate rows are removed first. If `order_item_id` is duplicated, the first complete record is retained after sorting by non-null count. This preserves the fact-table grain.

In [ ]:
rows_before, cols_before = df.shape
exact_dup_count = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
log_step('remove_exact_duplicates', rows_before, len(df), cols_before, df.shape[1], f'Removed {exact_dup_count} exact duplicates.')

if 'order_item_id' in df.columns:
    rows_before = len(df)
    df['_non_null_count'] = df.notna().sum(axis=1)
    df = (
        df.sort_values(['order_item_id', '_non_null_count'], ascending=[True, False])
          .drop_duplicates(subset=['order_item_id'], keep='first')
          .drop(columns='_non_null_count')
          .copy()
    )
    log_step('deduplicate_order_item_id', rows_before, len(df), cols_before, df.shape[1], 'Retained the most complete row per order_item_id.')

print(f'Rows after duplicate handling: {len(df):,}')

Rows after duplicate handling: 180,519


## 9. Missing-value strategy
No synthetic sales, profit, quantity, or delivery values are invented.

- Rows missing the primary fact key are removed.
- Text dimensions use `Unknown` only when a valid categorical member is needed.
- Customer ZIP code remains nullable because geography can still be represented by city/state/country.
- Optional columns with 100% missing values are removed.
- Critical missing values are reported for review rather than silently imputed.

In [ ]:
rows_before, cols_before = df.shape
all_missing_cols = df.columns[df.isna().all()].tolist()
if all_missing_cols:
    df = df.drop(columns=all_missing_cols)
log_step('drop_all_missing_columns', rows_before, len(df), cols_before, df.shape[1], f'Dropped all-null columns: {all_missing_cols}')

if 'order_item_id' in df.columns:
    rows_before = len(df)
    df = df[df['order_item_id'].notna()].copy()
    log_step('remove_missing_primary_key', rows_before, len(df), df.shape[1], df.shape[1], 'Removed rows without order_item_id.')

categorical_fill_cols = [
    'delivery_status', 'category_name', 'customer_city', 'customer_country',
    'customer_segment', 'customer_state', 'department_name', 'market',
    'order_city', 'order_country', 'order_region', 'order_state',
    'order_status', 'product_name', 'shipping_mode', 'payment_type'
]
for col in [c for c in categorical_fill_cols if c in df.columns]:
    df[col] = df[col].fillna('Unknown')

critical_cols = [c for c in [
    'order_item_id', 'order_id', 'customer_id', 'product_card_id',
    'order_date', 'sales', 'order_item_quantity', 'order_item_product_price'
] if c in df.columns]
critical_missing = df[critical_cols].isna().sum().rename('missing_count').to_frame()
critical_missing['missing_pct'] = (critical_missing['missing_count'] / len(df) * 100).round(4)
display(critical_missing)

log_step('handle_missing_values', len(df), len(df), df.shape[1], df.shape[1], 'Filled selected dimensions with Unknown; retained nullable analytical fields.')

,missing_count,missing_pct
order_item_id,0,0.0000
order_id,0,0.0000
customer_id,0,0.0000
product_card_id,0,0.0000
order_date,0,0.0000
sales,0,0.0000
order_item_quantity,0,0.0000
order_item_product_price,0,0.0000


## 10. Standardize categorical labels
Known labels are normalized without changing their business meaning.

In [ ]:
categorical_standardization = {
    'delivery_status': {
        'Advance shipping': 'Advance Shipping',
        'Late delivery': 'Late Delivery',
        'Shipping canceled': 'Shipping Canceled',
        'Shipping on time': 'Shipping On Time'
    },
    'shipping_mode': {
        'Standard Class': 'Standard Class',
        'First Class': 'First Class',
        'Second Class': 'Second Class',
        'Same Day': 'Same Day'
    }
}

for col, mapping in categorical_standardization.items():
    if col in df.columns:
        df[col] = df[col].replace(mapping)

# Title-case selected geographical text while preserving IDs and product names
for col in [c for c in ['customer_city','customer_country','customer_state','order_city','order_country','order_state'] if c in df.columns]:
    df[col] = df[col].where(df[col].eq('Unknown'), df[col].str.title())

log_step('standardize_categories', len(df), len(df), df.shape[1], df.shape[1], 'Normalized known categorical labels.')

## 11. Feature engineering
Features are designed for SQL and Tableau analysis while retaining original fields.

In [ ]:
# Delivery features
if {'days_for_shipping_real', 'days_for_shipment_scheduled'}.issubset(df.columns):
    df['shipping_delay_days'] = df['days_for_shipping_real'] - df['days_for_shipment_scheduled']
    df['delivery_performance'] = np.select(
        [df['shipping_delay_days'] > 0, df['shipping_delay_days'] == 0, df['shipping_delay_days'] < 0],
        ['Late', 'On Schedule', 'Early'],
        default='Unknown'
    )

if 'late_delivery_risk' in df.columns:
    df['is_late_delivery'] = df['late_delivery_risk'].eq(1).astype('Int8')

# Financial features
if {'sales', 'order_profit_per_order'}.issubset(df.columns):
    df['profit_margin_pct'] = np.where(
        df['sales'].ne(0), (df['order_profit_per_order'] / df['sales']) * 100, np.nan
    )
if {'order_item_discount', 'sales'}.issubset(df.columns):
    df['gross_sales_before_discount'] = df['sales'] + df['order_item_discount']
if 'order_profit_per_order' in df.columns:
    df['profitability_status'] = np.select(
        [df['order_profit_per_order'] > 0, df['order_profit_per_order'] == 0, df['order_profit_per_order'] < 0],
        ['Profitable', 'Break Even', 'Loss'], default='Unknown'
    )

# Time dimensions
if 'order_date' in df.columns:
    df['order_year'] = df['order_date'].dt.year.astype('Int64')
    df['order_quarter'] = ('Q' + df['order_date'].dt.quarter.astype('Int64').astype('string')).astype('string')
    df['order_month'] = df['order_date'].dt.month.astype('Int64')
    df['order_month_name'] = df['order_date'].dt.month_name()
    df['order_week'] = df['order_date'].dt.isocalendar().week.astype('Int64')
    df['order_day'] = df['order_date'].dt.day.astype('Int64')
    df['order_day_name'] = df['order_date'].dt.day_name()
    df['order_date_key'] = df['order_date'].dt.strftime('%Y%m%d').astype('Int64')

if 'shipping_date' in df.columns:
    df['shipping_date_key'] = df['shipping_date'].dt.strftime('%Y%m%d').astype('Int64')

# Round currency/ratio fields for stable export
for col in [c for c in [
    'order_profit','sales_per_customer','order_item_discount','order_item_discount_rate',
    'order_item_product_price','order_item_profit_ratio','sales','order_item_total',
    'order_profit_per_order','product_price','profit_margin_pct','gross_sales_before_discount'
] if c in df.columns]:
    df[col] = df[col].round(4)

log_step('feature_engineering', len(df), len(df), df.shape[1], df.shape[1], 'Added delivery, profitability, and date features.')
print('Current shape:', df.shape)
display(df.head())

Current shape: (180519, 59)


,payment_type,days_for_shipping,days_for_shipment,order_profit,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_id,customer_segment,customer_state,customer_zipcode,department_id,department_name,latitude,longitude,market,order_city,order_country,order_customer_id,order_date,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_state,order_status,order_zipcode,product_card_id,product_category_id,product_name,product_price,product_status,shipping_date,shipping_mode,is_late_delivery,profit_margin_pct,gross_sales_before_discount,profitability_status,order_year,order_quarter,order_month,order_month_name,order_week,order_day,order_day_name,order_date_key,shipping_date_key
33833,CASH,2,4,88.7900,239.9800,Advance Shipping,0,43,Camping & Hiking,Hickory,Ee. Uu.,11599,Consumer,Nc,"28,601.0000",7,Fan Shop,35.7767,-81.3626,LATAM,Mexico City,México,11599,2015-01-01 00:00:00,1,957,60.0000,0.2000,1,299.9800,0.3700,1,299.9800,239.9800,88.7900,Central America,Distrito Federal,CLOSED,NaN,957,43,Diamondback Women's Serene Classic Comfort Bi,299.9800,0,2015-01-03 00:00:00,Standard Class,0,29.5986,359.9800,Profitable,2015,Q1,1,January,1,1,Thursday,20150101,20150103
77011,PAYMENT,3,4,91.1800,193.9900,Advance Shipping,0,48,Water Sports,Chicago,Ee. Uu.,256,Consumer,Il,"60,625.0000",7,Fan Shop,41.8327,-87.9805,LATAM,Dos Quebradas,Colombia,256,2015-01-01 00:21:00,2,1073,6.0000,0.0300,2,199.9900,0.4700,1,199.9900,193.9900,91.1800,South America,Risaralda,PENDING_PAYMENT,NaN,1073,48,Pelican Sunstream 100 Kayak,199.9900,0,2015-01-04 00:21:00,Standard Class,0,45.5923,205.9900,Profitable,2015,Q1,1,January,1,1,Thursday,20150101,20150104
109322,PAYMENT,3,4,68.2500,227.5000,Advance Shipping,0,24,Women's Apparel,Chicago,Ee. Uu.,256,Consumer,Il,"60,625.0000",5,Golf,41.8327,-87.9805,LATAM,Dos Quebradas,Colombia,256,2015-01-01 00:21:00,2,502,22.5000,0.0900,3,50.0000,0.3000,5,250.0000,227.5000,68.2500,South America,Risaralda,PENDING_PAYMENT,NaN,502,24,Nike Men's Dri-FIT Victory Golf Polo,50.0000,0,2015-01-04 00:21:00,Standard Class,0,27.3000,272.5000,Profitable,2015,Q1,1,January,1,1,Thursday,20150101,20150104
87884,PAYMENT,3,4,36.4700,107.8900,Advance Shipping,0,18,Men's Footwear,Chicago,Ee. Uu.,256,Consumer,Il,"60,625.0000",4,Apparel,41.8327,-87.9805,LATAM,Dos Quebradas,Colombia,256,2015-01-01 00:21:00,2,403,22.1000,0.1700,4,129.9900,0.3400,1,129.9900,107.8900,36.4700,South America,Risaralda,PENDING_PAYMENT,NaN,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.9900,0,2015-01-04 00:21:00,Standard Class,0,28.0560,152.0900,Profitable,2015,Q1,1,January,1,1,Thursday,20150101,20150104
63764,CASH,5,4,4.1000,40.9800,Late Delivery,1,40,Accessories,San Antonio,Ee. Uu.,8827,Home Office,Tx,"78,240.0000",6,Outdoors,29.5200,-98.6374,LATAM,Dos Quebradas,Colombia,8827,2015-01-01 01:03:00,4,897,9.0000,0.1800,5,24.9900,0.1000,2,49.9800,40.9800,4.1000,South America,Risaralda,CLOSED,NaN,897,40,Team Golf New England Patriots Putter Grip,24.9900,0,2015-01-06 01:03:00,Standard Class,1,8.2033,58.9800,Profitable,2015,Q1,1,January,1,1,Thursday,20150101,20150106


## 12. Business-rule validation after cleaning

In [ ]:
validation_results = []

def validate(name, failure_mask, severity='high'):
    mask = pd.Series(failure_mask, index=df.index).fillna(False)
    failed = int(mask.sum())
    validation_results.append({
        'rule': name,
        'severity': severity,
        'failed_rows': failed,
        'failed_pct': round(failed/len(df)*100, 4),
        'status': 'PASS' if failed == 0 else 'REVIEW'
    })

if 'order_item_id' in df.columns:
    validate('order_item_id is not null', df['order_item_id'].isna())
    validate('order_item_id is unique', df['order_item_id'].duplicated(keep=False))
if 'order_item_quantity' in df.columns:
    validate('quantity > 0', df['order_item_quantity'] <= 0)
if 'sales' in df.columns:
    validate('sales >= 0', df['sales'] < 0)
if 'order_item_product_price' in df.columns:
    validate('product price >= 0', df['order_item_product_price'] < 0)
if 'order_item_discount_rate' in df.columns:
    validate('discount rate between 0 and 1', ~df['order_item_discount_rate'].between(0, 1, inclusive='both'))
if 'late_delivery_risk' in df.columns:
    validate('late_delivery_risk in {0,1}', ~df['late_delivery_risk'].isin([0, 1]))
if {'order_date','shipping_date'}.issubset(df.columns):
    validate('shipping date is not before order date', df['shipping_date'] < df['order_date'], severity='medium')
if 'days_for_shipping_real' in df.columns:
    validate('actual shipping days >= 0', df['days_for_shipping_real'] < 0)

validation_report = pd.DataFrame(validation_results)
display(validation_report)

# Hard assertions for structural requirements
assert len(df) > 0, 'Clean dataset is empty.'
assert not df.columns.duplicated().any(), 'Duplicate column names exist.'
assert df.duplicated().sum() == 0, 'Exact duplicates remain.'
if 'order_item_id' in df.columns:
    assert df['order_item_id'].notna().all(), 'Missing order_item_id remains.'
    assert df['order_item_id'].is_unique, 'order_item_id is not unique.'
print('Structural assertions passed.')

,rule,severity,failed_rows,failed_pct,status
0,order_item_id is not null,high,0,0.0000,PASS
1,order_item_id is unique,high,0,0.0000,PASS
2,quantity > 0,high,0,0.0000,PASS
3,sales >= 0,high,0,0.0000,PASS
4,product price >= 0,high,0,0.0000,PASS
5,discount rate between 0 and 1,high,0,0.0000,PASS
6,"late_delivery_risk in {0,1}",high,0,0.0000,PASS
7,shipping date is not before order date,medium,0,0.0000,PASS


Structural assertions passed.


## 13. Reconciliation and readiness score

In [ ]:
clean_rows, clean_cols = df.shape
reconciliation = pd.DataFrame({
    'metric': ['raw_rows','clean_rows','rows_removed','retention_rate_pct','raw_columns','clean_columns'],
    'value': [
        raw_rows, clean_rows, raw_rows-clean_rows,
        round(clean_rows/raw_rows*100, 4), raw_cols, clean_cols
    ]
})
display(reconciliation)

audit_df = pd.DataFrame(audit_log)
display(audit_df)

completeness = (1 - df.isna().sum().sum()/df.size) * 100
uniqueness = (1 - df.duplicated().sum()/len(df)) * 100
validity = (1 - validation_report['failed_rows'].sum()/(len(df)*max(len(validation_report),1))) * 100
readiness_score = 0.35*completeness + 0.25*uniqueness + 0.40*max(0, validity)

print(f'Completeness : {completeness:.2f}%')
print(f'Uniqueness   : {uniqueness:.2f}%')
print(f'Validity     : {validity:.2f}%')
print(f'READINESS    : {readiness_score:.2f}/100')

,metric,value
0,raw_rows,"180,519.0000"
1,clean_rows,"180,519.0000"
2,rows_removed,0.0000
3,retention_rate_pct,100.0000
4,raw_columns,53.0000
5,clean_columns,59.0000


,step,rows_before,rows_after,rows_removed,columns_before,columns_after,note
0,load_raw_data,180519,180519,0,53,53,Raw CSV loaded successfully.
1,standardize_column_names,180519,180519,0,53,53,Converted columns to snake_case.
2,drop_pii_and_unused_columns,180519,180519,0,53,46,"Dropped: ['customer_email', 'customer_fname', ..."
3,normalize_text,180519,180519,0,46,46,Trimmed whitespace and normalized blank strings.
4,standardize_data_types,180519,180519,0,46,46,Parsed dates and converted numeric fields.
5,remove_exact_duplicates,180519,180519,0,46,46,Removed 0 exact duplicates.
6,deduplicate_order_item_id,180519,180519,0,46,46,Retained the most complete row per order_item_id.
7,drop_all_missing_columns,180519,180519,0,46,46,Dropped all-null columns: []
8,remove_missing_primary_key,180519,180519,0,46,46,Removed rows without order_item_id.
9,handle_missing_values,180519,180519,0,46,46,Filled selected dimensions with Unknown; retai...


Completeness : 98.54%
Uniqueness   : 100.00%
Validity     : 100.00%
READINESS    : 99.49/100


## 14. Export clean dataset and documentation artifacts
Dates are exported in MySQL-friendly format. The output row count is read back to verify that the file is not truncated.

In [ ]:
OUTPUT_DIR = Path('/content/nexora_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / 'dataco_supply_chain_cleaned.csv'

# MySQL-friendly timestamp strings in exported copy
export_df = df.copy()
for col in [c for c in ['order_date','shipping_date'] if c in export_df.columns]:
    export_df[col] = export_df[col].dt.strftime('%Y-%m-%d %H:%M:%S')

export_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
column_mapping.to_csv(OUTPUT_DIR / 'column_name_mapping.csv', index=False)
audit_df.to_csv(OUTPUT_DIR / 'cleaning_audit_log.csv', index=False)
validation_report.to_csv(OUTPUT_DIR / 'cleaning_validation_report.csv', index=False)
reconciliation.to_csv(OUTPUT_DIR / 'row_reconciliation.csv', index=False)

# Lightweight read-back validation—counts every data row without loading the full file again
with open(OUTPUT_CSV, 'r', encoding='utf-8') as f:
    exported_line_count = sum(1 for _ in f) - 1

file_size_mb = OUTPUT_CSV.stat().st_size / 1024**2
print(f'Output file       : {OUTPUT_CSV}')
print(f'Exported rows     : {exported_line_count:,}')
print(f'Expected rows     : {len(export_df):,}')
print(f'Exported columns  : {export_df.shape[1]}')
print(f'File size         : {file_size_mb:,.2f} MB')

assert exported_line_count == len(export_df), 'CSV export appears truncated.'
print('CSV read-back row-count validation passed.')

Output file       : /content/nexora_outputs/dataco_supply_chain_cleaned.csv
Exported rows     : 180,519
Expected rows     : 180,519
Exported columns  : 59
File size         : 76.85 MB
CSV read-back row-count validation passed.


## 15. Optional download in Google Colab

In [ ]:
# Uncomment in Google Colab to download the cleaned CSV.
# from google.colab import files
# files.download(str(OUTPUT_CSV))

print('Files created:')
for p in sorted(OUTPUT_DIR.iterdir()):
    print('-', p.name)

Files created:
- cleaning_audit_log.csv
- cleaning_validation_report.csv
- column_name_mapping.csv
- dataco_supply_chain_cleaned.csv
- row_reconciliation.csv


## Completion checklist
- [ ] Raw file loaded with approximately 180,519 rows and 53 columns
- [ ] Column names converted to snake_case
- [ ] Direct personal data removed
- [ ] Dates and numeric fields standardized
- [ ] Exact duplicates removed
- [ ] `order_item_id` is non-null and unique
- [ ] Missing-value strategy documented
- [ ] Outliers retained unless a business rule proves them invalid
- [ ] Delivery, profitability, and time features created
- [ ] Validation report reviewed
- [ ] Export row count matches DataFrame row count
- [ ] `dataco_supply_chain_cleaned.csv` created successfully

### Expected primary output
`/content/nexora_outputs/dataco_supply_chain_cleaned.csv`

This CSV is ready for the next project phase: **MySQL staging-table creation and bulk import**.